# Sample Data Extraction

**Aim**
* extract a range of data from Databricks for each of v2 providers, and analyse distribution across multiple key features

In [ ]:
from dotenv import load_dotenv
import os, gzip
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import pandas as pd
from databricks import sql
import os
import plotly.express as px
import yaml
import re
import plotly.express as px

### Functions

In [ ]:
def parse_yaml_lenient(text: str):
    try:
        # strip out ruby-specific tags
        cleaned = re.sub(r'!ruby/object:[^\s]+\s+', '', text)
        return yaml.safe_load(cleaned) or {}
    except Exception:
        return {}

def flatten_yaml_column(df: pd.DataFrame, col: str, prefix: str = "") -> pd.DataFrame:
    expanded = df[col].apply(parse_yaml_lenient).apply(pd.Series)
    if prefix:
        expanded = expanded.add_prefix(prefix)
    return pd.concat([df, expanded], axis=1)

In [ ]:
def plot_value_counts(series, title=None, width=None, height=None, show_table=False, table_rows=None, top_n=10, horizontal=False):
    if title is None:
        title = f"Value Counts of {series.name}"
    
    counts = series.value_counts().reset_index()
    counts.columns = ["category", "count"]
    
    print(f"Total distinct values: {len(counts)}")
    
    if show_table:
        if table_rows is not None:
            display(counts.head(table_rows))
        else:
            with pd.option_context("display.max_rows", None):
                display(counts)
    
    counts = counts.head(top_n)
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)
    counts["label"] = counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)
    
    if horizontal:
        counts = counts.iloc[::-1]
        fig = px.bar(counts, x="count", y="category", title=title, text="label", orientation="h")
    else:
        fig = px.bar(counts, x="category", y="count", title=title, text="label")
        fig.update_layout(xaxis_tickangle=-45)
    
    fig.update_layout(width=width, height=height)
    fig.update_traces(textposition="outside")
    fig.show()

In [ ]:
from ipywidgets import interact, Dropdown

def plot_value_counts_by_provider(df, column, provider_column="provider_name", title=None, width=None, height=None, show_table=False, table_rows=None, top_n=10, horizontal=False):
    
    providers = ["All"] + sorted(df[provider_column].dropna().unique().tolist())
    
    def _plot(provider):
        if provider == "All":
            series = df[column]
        else:
            series = df[df[provider_column] == provider][column]
        
        chart_title = title or f"Value Counts of {column}"
        chart_title = f"{chart_title} (Provider: {provider})"
        
        counts = series.value_counts().reset_index()
        counts.columns = ["category", "count"]
        
        print(f"Total distinct values: {len(counts)}")
        
        if show_table:
            if table_rows is not None:
                display(counts.head(table_rows))
            else:
                with pd.option_context("display.max_rows", None):
                    display(counts)
        
        counts = counts.head(top_n)
        counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)
        counts["label"] = counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)
        
        if horizontal:
            counts = counts.iloc[::-1]
            fig = px.bar(counts, x="count", y="category", title=chart_title, text="label", orientation="h")
        else:
            fig = px.bar(counts, x="category", y="count", title=chart_title, text="label")
            fig.update_layout(xaxis_tickangle=-45)
        
        fig.update_layout(width=width, height=height)
        fig.update_traces(textposition="outside")
        fig.show()
    
    interact(_plot, provider=Dropdown(options=providers, value="All", description="Provider:"))

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown, Output
from IPython.display import display, clear_output

def plot_value_counts_by_provider(df, column, provider_column="provider_name", title=None,
                                   width=None, height=None, show_table=False, table_rows=None,
                                   top_n=10, horizontal=False, facet=False, facet_cols=3, facet_font_size=9,
                                   subplot_width=500, subplot_height=600):

    if facet:
        _plot_facet(df, column, provider_column, title, width, height, top_n, horizontal, facet_cols, facet_font_size, subplot_width, subplot_height)
        return

    providers = ["All"] + sorted(df[provider_column].dropna().unique().tolist())
    out = Output()
    display(out)

    def _plot(provider):
        with out:
            clear_output(wait=True)

            if provider == "All":
                series = df[column]
            else:
                series = df[df[provider_column] == provider][column]

            chart_title = title or f"Value Counts of {column}"
            chart_title = f"{chart_title} (Provider: {provider})"

            counts = series.value_counts().reset_index()
            counts.columns = ["category", "count"]

            print(f"Total distinct values: {len(counts)}")

            if show_table:
                if table_rows is not None:
                    display(counts.head(table_rows))
                else:
                    with pd.option_context("display.max_rows", None):
                        display(counts)

            counts = counts.head(top_n)
            counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)
            counts["label"] = counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)

            if horizontal:
                counts = counts.iloc[::-1]
                fig = px.bar(counts, x="count", y="category", title=chart_title, text="label", orientation="h")
            else:
                fig = px.bar(counts, x="category", y="count", title=chart_title, text="label")
                fig.update_layout(xaxis_tickangle=-45)

            fig.update_layout(width=width, height=height)
            fig.update_traces(textposition="outside")
            fig.show()

    interact(_plot, provider=Dropdown(options=providers, value="All", description="Provider:"))


def _plot_facet(df, column, provider_column, title, width, height, top_n, horizontal, facet_cols, facet_font_size, subplot_width, subplot_height):
    providers = sorted(df[provider_column].dropna().unique().tolist())
    n = len(providers)
    facet_rows = -(-n // facet_cols)  # ceiling division

    total_width = width or subplot_width * facet_cols
    total_height = height or subplot_height * facet_rows

    fig = make_subplots(
        rows=facet_rows, cols=facet_cols,
        subplot_titles=providers,
        horizontal_spacing=0.03,
        vertical_spacing=0.06
    )

    for i, provider in enumerate(providers):
        row = i // facet_cols + 1
        col = i % facet_cols + 1

        series = df[df[provider_column] == provider][column]
        counts = series.value_counts().reset_index()
        counts.columns = ["category", "count"]
        counts = counts.head(top_n)
        counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)
        counts["label"] = counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)

        if horizontal:
            counts = counts.iloc[::-1]
            fig.add_trace(
                go.Bar(x=counts["count"], y=counts["category"].astype(str),
                       text=counts["label"], textposition="outside",
                       orientation="h", showlegend=False),
                row=row, col=col
            )
        else:
            fig.add_trace(
                go.Bar(x=counts["category"].astype(str), y=counts["count"],
                       text=counts["label"], textposition="outside",
                       showlegend=False),
                row=row, col=col
            )
            fig.update_xaxes(tickangle=-45, row=row, col=col)

    chart_title = title or f"Value Counts of {column} by Provider"
    fig.update_layout(title=chart_title, width=total_width, height=total_height,
                      font=dict(size=facet_font_size))
    fig.update_annotations(font_size=facet_font_size + 2)  # subplot titles slightly larger
    fig.show()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
load_dotenv(dotenv_path=r"/home/sagemaker-user/swtest1/llm_poc/.env", override=True)

In [ ]:
print("HOST:", os.getenv("DATABRICKS_SERVER_HOSTNAME"))
print("HTTP PATH:", os.getenv("DATABRICKS_HTTP_PATH"))
print("TOKEN set?", bool(os.getenv("DATABRICKS_TOKEN")))

In [ ]:
def dbx_query_df(query: str, *, catalog=None, schema=None) -> pd.DataFrame:
    with sql.connect(
        server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
        http_path=os.getenv("DATABRICKS_HTTP_PATH"),
        access_token=os.getenv("DATABRICKS_TOKEN"),
        catalog=catalog or os.getenv("DATABRICKS_CATALOG"),
        schema=schema  or os.getenv("DATABRICKS_SCHEMA"),
        use_cloud_fetch=True,
    ) as conn, conn.cursor() as cur:
        cur.execute(query)
        return cur.fetchall_arrow().to_pandas(types_mapper=pd.ArrowDtype)


def dbx_scalar(query: str, params=None, *, catalog=None, schema=None):
    with sql.connect(
        server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
        http_path=os.getenv("DATABRICKS_HTTP_PATH"),
        access_token=os.getenv("DATABRICKS_TOKEN"),
        catalog=catalog or os.getenv("DATABRICKS_CATALOG"),
        schema=schema  or os.getenv("DATABRICKS_SCHEMA"),
        use_cloud_fetch=True,
    ) as conn, conn.cursor() as cur:
        cur.execute(query, params or ())
        return cur.fetchone()[0]  # first column of the first row

In [ ]:
TARGET_PER_PROVIDER = 15000  # grab generously, trim later

df_raw = dbx_query_df(f"""
SELECT 
    t.id                            AS transaction_id,
    pa.user_id,
    p.name                          AS provider_name,
    p.aggregator,
    p.external_id,
    t.account_id,
    t.container,
    a.account_type,
    t.is_detail_available,
    t.transaction_date,
    t.original_description,
    t.amount,
    t.base_category_key,
    t.transaction_type,
    t.cdr_merchant_category_code    AS merchant_category_code,
    t.cdr_biller_code,
    t.cdr_biller_name,
    t.merchant_name,
    m.name                          AS frollo_merchant,
    t.properties
FROM prod.public.accounts_silver_view a
INNER JOIN prod.public.provider_accounts_silver_view pa ON pa.id = a.provider_account_id
INNER JOIN prod.public.providers_silver_view p ON p.id = pa.provider_id
INNER JOIN prod.public.transactions_silver_view t ON t.account_id = a.id
INNER JOIN prod.public.merchants_silver_view m ON m.id = t.merchant_id
WHERE p.ideas_enrich_v2 = True
  AND t.created_at BETWEEN DATE '2025-09-01' AND DATE '2025-12-31'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.name
    ORDER BY RAND(42)
) <= {TARGET_PER_PROVIDER}
""")

In [ ]:
df_raw.base_category_key.value_counts()

In [ ]:
df_raw.shape

In [ ]:
print(f"Extracted: {len(df_raw):,} transactions")
print(f"Providers: {df_raw['provider_name'].nunique()}")
print(f"\nPer-provider counts:")
print(df_raw['provider_name'].value_counts().to_string())

In [ ]:
df_raw = flatten_yaml_column(df_raw, "properties", prefix="prop_")

In [ ]:
df_raw.to_parquet(
    "data/raw_data2_large.parquet",
    engine="pyarrow",       # or "fastparquet"
    compression="snappy",   # common: "snappy", "gzip", "zstd", or None
    index=False             # often desirable to avoid saving the index
)


### 0. read data from file

In [ ]:
#df_raw = pd.read_parquet("data/raw_data2_large.parquet")

In [ ]:
df_raw.columns

In [ ]:
def heatmap_crosstab(df, index_col, value_col, show_pct=True, height=600, label_map=None):
    ct = pd.crosstab(df[index_col], df[value_col])

    # Apply label mapping to columns if provided
    if label_map:
        ct = ct.rename(columns={k: f"{k} ({v})" for k, v in label_map.items()})

    data = ct.div(ct.sum(axis=1), axis=0) * 100 if show_pct else ct

    fig = px.imshow(
        data,
        text_auto='.1f' if show_pct else 'd',
        color_continuous_scale='YlOrRd',
        labels={'color': '%' if show_pct else 'count'},
        title=f'{value_col} by {index_col} ({"%" if show_pct else "counts"})',
        height=height,
        width=800,
        aspect='auto',  # let it stretch to fill the width
    )
    fig.update_xaxes(side='bottom', dtick=1, tickangle=270)  # dtick=1 forces every label to show
    fig.update_coloraxes(showscale=False)  
    fig.show()

# Transaction type mapping
TXN_TYPE_MAP = {
    0: 'fee',
    1: 'interest_charged',
    2: 'interest_paid',
    3: 'transfer_outgoing',
    4: 'transfer_incoming',
    5: 'payment',
    6: 'direct_debit',
    7: 'other',
}
# Usage
#heatmap_crosstab(df, 'provider_name', 'transaction_type')
#heatmap_crosstab(df, 'provider_name', 'category_type')

In [ ]:
heatmap_crosstab(df_raw, 'provider_name', 'transaction_type', label_map=TXN_TYPE_MAP)

In [ ]:
pd.crosstab(df_raw['provider_name'], df_raw['transaction_type'])

In [ ]:
df_raw.head().T

### 1. BC (spend / non-spend) Distribution

In [ ]:
plot_value_counts(df_raw.prop_ideas_has_merch, width=800, height=500, show_table=True)

### 2. Category Type Distribution (Non-Spend)

In [ ]:
plot_value_counts(df_raw[df_raw.prop_ideas_has_merch=="0"].prop_ideas_categorisation_name_nonspend, width=800, height=500, show_table=True, top_n=20, horizontal=True, 
    title = "Category Type Distribution (Non-Spend)")

### 3. Transfers - base category key distribution

In [ ]:
plot_value_counts(df_raw[df_raw.prop_ideas_categorisation_name_nonspend=="TRANSFERS"].base_category_key, width=800, height=800, show_table=True, top_n=30, horizontal=True, table_rows=20,
    title = "Transfers - base category key distribution")

### 4. Spend - base category key distribution

In [ ]:
plot_value_counts(df_raw[df_raw.prop_ideas_has_merch=="1"].base_category_key, width=800, height=800, show_table=True, top_n=30, horizontal=True, table_rows=20,
    title = "Spend - base category key distribution")

### 5. Non-Spend - base category key distribution

In [ ]:
plot_value_counts(df_raw[df_raw.prop_ideas_has_merch=="0"].base_category_key, width=800, height=800, show_table=True, top_n=30, horizontal=True, table_rows=20,
    title = "Non-Spend - base category key distribution")

In [ ]:
pd.set_option('display.max_colwidth', 200)

### 6. Confirm which merchant field to check

In [ ]:
df_raw.columns

* what's the difference between frollo_merchant and ideas_merchant_nme?

In [ ]:
#df_raw.loc[df_raw.merchant_name.notna() or df_raw.prop_ideas_merchant_name.notna(), ['provider_name', 'original_description','base_category_key', 'merchant_name', 'frollo_merchant', 'prop_ideas_merchant_name', 'prop_ner_entity', 'prop_ner_entity_original']]
df_raw.loc[df_raw.merchant_name.notna() | df_raw.prop_ideas_merchant_name.notna(), ['provider_name', 'original_description','base_category_key', 'merchant_name', 'frollo_merchant', 'prop_ideas_merchant_name', 'prop_ner_entity', 'prop_ner_entity_original']].sample(20)

# use prop_ideas_merchant_name as the target


In [ ]:
df_raw.head().T

### 7. Per-Provider analysis

#### a. BC Distribution

In [ ]:
plot_value_counts_by_provider(df_raw, "prop_ideas_has_merch", width=800, height=500)

In [ ]:
plot_value_counts_by_provider(df_raw, "prop_ideas_has_merch", facet=True, facet_cols=3, horizontal=True, subplot_width=350, subplot_height=350)

#### b. Category Type Distribution

In [ ]:
plot_value_counts_by_provider(df_raw[df_raw.prop_ideas_has_merch=="0"], "prop_ideas_categorisation_name_nonspend", top_n=20, 
    horizontal=True, facet=True, facet_font_size=8,
    title = "Category Type Distribution (Non-Spend)", facet_cols=3, subplot_width=350, subplot_height=400)


#### c. Transfers - base category key Distribution

In [ ]:
plot_value_counts_by_provider(df_raw[df_raw.prop_ideas_categorisation_name_nonspend=="TRANSFERS"], "base_category_key", top_n=20, 
    horizontal=True, facet=True, facet_font_size=8,
    title = "Tranfers - Base Category Key Distribution", facet_cols=3, subplot_width=350, subplot_height=400)


#### d. Spend - Base Category Key

In [ ]:
plot_value_counts_by_provider(df_raw[df_raw.prop_ideas_has_merch=="1"], "base_category_key", top_n=20, 
    horizontal=True, facet=True, facet_font_size=8,
    title = "Spend - Base Category Key Distribution", facet_cols=3, subplot_width=350, subplot_height=400)

##### e. Non-Spend - Base Category Key

In [ ]:
plot_value_counts_by_provider(df_raw[df_raw.prop_ideas_has_merch=="0"], "base_category_key", top_n=20, 
    horizontal=True, facet=True, facet_font_size=8,
    title = "Non-Spend - Base Category Key Distribution", facet_cols=3, subplot_width=350, subplot_height=400)

### 8. Include Edge Cases - Known errors

1. identify 14 v2 providers -13 from shared
2. explore distribution of various key fields
3. which merchant name field to analyse?

1. Query Design
Write Databricks SQL/PySpark query for transaction extraction

Include fields: transaction_id, description, transaction_type, account_type, provider, amount, MCC, timestamp

Include production labels: merchant_name, spend_or_non_spend, category_type, base_category_key

Add stratified sampling logic for balanced coveragem

2. Provider Coverage Strategy
List all 14 V2 providers with target transaction counts

Identify key V1 providers for inclusion

Ensure minimum 200 transactions per provider

Include proportional category distribution